In [ ]:
#cutoff

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix

# ================= CONFIG =================
LABEL_CSV = "../LiverFibrosisCohort/LiverFibrosisCohort_MasterTable_Proccessed_527.csv"
FEATURE_ROOT = "../LiverFibrosisCohort/deep_medsiglip_v2"
OUTPUT_CSV = "../Results/deep_mlp_results_with_sen_spe.csv"

AUC_PLOT_PATH = "../Results/deep_plots/deep_features_combined_auc.png"
BAL_ACC_PLOT_PATH = "../Results/deep_plots/deep_features_combined_balanced_accuracy.png"

MODALITIES = ["ADC", "T1", "T2"]
PLOT_MODALITY_ORDER = ["T1", "T2", "ADC"]
DEV_SITES = ["CCHMC", "WISC", "MICH"]

ID_COLUMN = "Image ID"
LABEL_COLUMN = "Fibrosis_Label"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 50
LR = 1e-3
SEED = 42
HIDDEN_SIZES = [128, 64, 32, 16, 1]

torch.manual_seed(SEED)
np.random.seed(SEED)

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# ================= ID =================
def normalize_id(pid):
    pid = str(pid)

    m = re.match(r"PT-(\d{3}-\d{4})-", pid)
    if m:
        return m.group(1)

    m = re.match(r"PT-(M\d+)-", pid)
    if m:
        return m.group(1)

    return pid


def get_site(pid):
    pid = str(pid)

    if re.match(r"^\d{3}-\d{4}$", pid):
        return "CCHMC"
    elif pid.startswith("GJ"):
        return "WISC"
    elif pid.startswith("C_"):
        return "NYU"
    elif re.match(r"^\d+$", pid) or pid.startswith("M"):
        return "MICH"
    else:
        return "UNKNOWN"


# ================= LABEL =================
def binarize_fibrosis(x):
    if pd.isna(x):
        return np.nan

    x = str(x)
    m = re.search(r"\d+", x)
    if not m:
        return np.nan

    val = int(m.group())

    if val in [0, 1]:
        return 0
    elif val in [2, 3, 4]:
        return 1
    return np.nan


# ================= DISPLAY HELPERS =================
def modality_display_name(modality):
    return {
        "T1": "T1w",
        "T2": "T2w",
        "ADC": "DWI"
    }.get(modality, modality)


def metric_display_name(metric):
    return {
        "auc": "AUROC",
        "balanced_accuracy": "Balanced Accuracy"
    }.get(metric, metric)


# ================= LOAD LABELS =================
labels_df = pd.read_csv(LABEL_CSV)
labels_df = labels_df.rename(columns={ID_COLUMN: "id"})

labels_df["id"] = labels_df["id"].astype(str).apply(normalize_id)
labels_df["label"] = labels_df[LABEL_COLUMN].apply(binarize_fibrosis)

labels_df = labels_df.dropna(subset=["label"])
labels_df["label"] = labels_df["label"].astype(int)
labels_df["site"] = labels_df["id"].apply(get_site)

labels_df = labels_df[["id", "label", "site"]]

# DEV cohort
labels_df = labels_df[labels_df["site"].isin(DEV_SITES)].copy()

print(f"DEV size: {len(labels_df)}")
print(labels_df["label"].value_counts())


# ================= MODEL =================
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()

        if hidden_dim == 1:
            self.net = nn.Linear(in_dim, 1)
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1)
            )

    def forward(self, x):
        return self.net(x).squeeze(1)


# ================= TRAIN =================
def train_eval(X, y, hidden_dim):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    all_probs = np.zeros(len(y), dtype=float)

    for tr, val in skf.split(X, y):
        X_tr, X_val = X[tr], X[val]
        y_tr, y_val = y[tr], y[val]

        X_tr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
        y_tr = torch.tensor(y_tr, dtype=torch.float32).to(DEVICE)
        X_val = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)

        model = MLP(X.shape[1], hidden_dim).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.BCEWithLogitsLoss()

        for _ in range(EPOCHS):
            model.train()
            logits = model(X_tr)
            loss = loss_fn(logits, y_tr)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(X_val)).cpu().numpy()

        all_probs[val] = probs

    preds = (all_probs > 0.5).astype(int)

    bal_acc = balanced_accuracy_score(y, preds)
    auc = roc_auc_score(y, all_probs)

    tn, fp, fn, tp = confusion_matrix(y, preds, labels=[0, 1]).ravel()
    sen = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spe = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return bal_acc, auc, sen, spe


def save_combined_barplot(results_df, metric, output_path, modality_order=None):
    if modality_order is None:
        modality_order = ["T1", "T2", "ADC"]

    fig, axes = plt.subplots(1, len(modality_order), figsize=(21, 6), sharey=True)

    if len(modality_order) == 1:
        axes = [axes]

    panel_labels = ["(a)", "(b)", "(c)"]
    metric_title = metric_display_name(metric)

    # ---- compute one shared upper y-limit across all subplots ----
    all_vals = []
    for modality in modality_order:
        sub = results_df[results_df["modality"] == modality].copy()
        if not sub.empty:
            all_vals.extend(sub[metric].dropna().tolist())

    if len(all_vals) == 0:
        print(f"No valid values found for metric={metric}. Skipping plot.")
        plt.close()
        return

    shared_upper = max(0.75, float(np.nanmax(all_vals)) + 0.08)
    shared_upper = min(1.0, shared_upper)

    for i, (ax, modality) in enumerate(zip(axes, modality_order)):
        sub = results_df[results_df["modality"] == modality].copy()

        if sub.empty:
            ax.axis("off")
            continue

        sub["hidden_dim"] = pd.Categorical(
            sub["hidden_dim"],
            categories=HIDDEN_SIZES,
            ordered=True
        )
        sub = sub.sort_values("hidden_dim")

        x_labels = [str(h) for h in sub["hidden_dim"].tolist()]
        y_vals = sub[metric].values

        bars = ax.bar(x_labels, y_vals, width=0.35)

        for bar, val in zip(bars, y_vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.008,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=12
            )

        ax.set_title(
            f"{panel_labels[i]} {modality_display_name(modality)} (Deep Features)",
            fontsize=22,
            pad=16
        )
        ax.set_xlabel("Hidden Dimension", fontsize=16)
        ax.tick_params(axis="x", labelsize=13)
        ax.set_ylim(0.0, shared_upper)

        # only keep y-axis label/ticks on the left subplot
        if i == 0:
            ax.set_ylabel(metric_title, fontsize=18)
            ax.tick_params(axis="y", labelsize=13)
        else:
            ax.tick_params(axis="y", left=False, labelleft=False)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved plot to: {output_path}")

# ================= RUN =================
results = {m: {"auc": [], "bal_acc": [], "sen": [], "spe": []} for m in MODALITIES}
rows = []

for modality in MODALITIES:
    print(f"\n===== {modality} =====")

    feat_path = os.path.join(FEATURE_ROOT, f"{modality}_medsiglip.csv")
    feat_df = pd.read_csv(feat_path)

    feat_df["id"] = feat_df["id"].astype(str).apply(normalize_id)

    merged = labels_df.merge(feat_df, on="id", how="inner")

    print(f"Matched: {len(merged)}")

    if len(merged) == 0:
        print(f"⚠️ Skipping {modality}")
        continue

    feature_cols = [c for c in merged.columns if c.startswith("medsiglip_")]

    X = merged[feature_cols].values
    y = merged["label"].values

    print(f"{modality} feature shape: {X.shape}")

    for h in HIDDEN_SIZES:
        bal_acc, auc, sen, spe = train_eval(X, y, h)

        print(
            f"MLP ({X.shape[1]} → {h} → 1): "
            f"Balanced Acc = {bal_acc:.4f}, "
            f"AUC = {auc:.4f}, "
            f"Sensitivity = {sen:.4f}, "
            f"Specificity = {spe:.4f}"
        )

        results[modality]["auc"].append(auc)
        results[modality]["bal_acc"].append(bal_acc)
        results[modality]["sen"].append(sen)
        results[modality]["spe"].append(spe)

        rows.append({
            "modality": modality,
            "n_features": X.shape[1],
            "hidden_dim": h,
            "balanced_accuracy": bal_acc,
            "auc": auc,
            "sensitivity": sen,
            "specificity": spe
        })

# ================= SAVE CSV =================
results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved results to: {OUTPUT_CSV}")
print(results_df)

# ================= SAVE COMBINED PLOTS =================
save_combined_barplot(
    results_df=results_df,
    metric="auc",
    output_path=AUC_PLOT_PATH,
    modality_order=PLOT_MODALITY_ORDER
)

save_combined_barplot(
    results_df=results_df,
    metric="balanced_accuracy",
    output_path=BAL_ACC_PLOT_PATH,
    modality_order=PLOT_MODALITY_ORDER
)

DEV size: 481
label
1    259
0    222
Name: count, dtype: int64

===== ADC =====
Matched: 230
ADC feature shape: (230, 1152)
MLP (1152 → 128 → 1): Balanced Acc = 0.6246, AUC = 0.6737, Sensitivity = 0.8692, Specificity = 0.3800
MLP (1152 → 64 → 1): Balanced Acc = 0.5792, AUC = 0.6661, Sensitivity = 0.9385, Specificity = 0.2200
MLP (1152 → 32 → 1): Balanced Acc = 0.5269, AUC = 0.6579, Sensitivity = 0.9538, Specificity = 0.1000
MLP (1152 → 16 → 1): Balanced Acc = 0.5000, AUC = 0.6572, Sensitivity = 1.0000, Specificity = 0.0000
MLP (1152 → 1 → 1): Balanced Acc = 0.5100, AUC = 0.6685, Sensitivity = 1.0000, Specificity = 0.0200

===== T1 =====
Matched: 340
T1 feature shape: (340, 1152)
MLP (1152 → 128 → 1): Balanced Acc = 0.5948, AUC = 0.6710, Sensitivity = 0.7647, Specificity = 0.4248
MLP (1152 → 64 → 1): Balanced Acc = 0.5853, AUC = 0.6669, Sensitivity = 0.8503, Specificity = 0.3203
MLP (1152 → 32 → 1): Balanced Acc = 0.5606, AUC = 0.6582, Sensitivity = 0.8663, Specificity = 0.2549
MLP (11